# 10.1 Fast scenario reproduction

Re-run the six historical Notebook 10 scenario definitions with the preserved historical optimizer adapter. The full-profile, seed-42 result is compared with Notebook 10's saved summary; the `test` profile is only an executable smoke test and cannot pass the scientific reproduction gate.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from estonia_landuse.sensitivity.benchmark import benchmark_manifest
from estonia_landuse.sensitivity.reproduction import (
    compare_reference_summary,
)
from estonia_landuse.sensitivity.runner import run_manifest
from estonia_landuse.sensitivity.sampling import (
    build_baseline_manifest,
    manifest_run_count,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
HISTORICAL_ROOT = (
    PROJECT_ROOT.parent.parent
    if PROJECT_ROOT.parent.name == ".worktrees"
    else PROJECT_ROOT
)

PROFILE = os.environ.get("SENSITIVITY_PROFILE", "full")
N_WORKERS = int(os.environ.get("SENSITIVITY_N_WORKERS", str(max(1, os.cpu_count() or 1))))
OUTPUT_ROOT = Path(
    os.environ.get(
        "SENSITIVITY_OUTPUT_ROOT",
        PROJECT_ROOT / "data/processed/legacy_sensitivity",
    )
).resolve()
FEATURES_PATH = Path(
    os.environ.get(
        "SENSITIVITY_FEATURES_PATH",
        HISTORICAL_ROOT / "data/processed/learned_carbon" / "features_with_forest.parquet",
    )
).resolve()
GRID_PATH = Path(
    os.environ.get(
        "SENSITIVITY_GRID_PATH",
        HISTORICAL_ROOT / "data/processed/v1" / "base_grid.gpkg",
    )
).resolve()
REFERENCE_SUMMARY_PATH = Path(
    os.environ.get(
        "SENSITIVITY_REFERENCE_SUMMARY_PATH",
        HISTORICAL_ROOT / "data/processed/learned_carbon" / "scenario_summary.parquet",
    )
).resolve()

if PROFILE not in {"test", "screen", "full"}:
    raise ValueError(f"Unknown SENSITIVITY_PROFILE: {PROFILE!r}")
if N_WORKERS < 1:
    raise ValueError("SENSITIVITY_N_WORKERS must be a positive integer")

print(f"Profile: {PROFILE}")
print(f"Workers: {N_WORKERS}")
print(f"Artifact root: {OUTPUT_ROOT}")

## Input validation

In [ ]:
NOTEBOOK_10_FEATURE_COLUMNS = [
    "urban_pct",
    "agriculture_pct",
    "grassland_pct",
    "forest_pct",
    "wetland_pct",
    "water_pct",
    "naturalness_score",
    "carbon_score",
    "protected_overlap_pct",
    "wetland_suitability",
    "biodiversity_proxy",
    "opportunity_cost_proxy",
    "rohemeeter_norm",
]

if PROFILE == "test":
    n_test_cells = 12
    position = np.linspace(0.0, 1.0, n_test_cells)
    context = pd.DataFrame(
        {
            "cell_id": np.arange(1, n_test_cells + 1),
            "forest_pct": 0.35 + 0.03 * position,
            "wetland_pct": 0.10 + 0.02 * position,
            "agriculture_pct": 0.30 - 0.03 * position,
            "grassland_pct": 0.15 - 0.02 * position,
            "urban_pct": np.full(n_test_cells, 0.05),
            "water_pct": np.full(n_test_cells, 0.05),
            "protected_overlap_pct": 0.05 * position,
            "wetland_suitability": 0.2 + 0.6 * position,
            "opportunity_cost_proxy": 0.1 + 0.5 * position,
            "predicted_tco2_ha_yr": 2.5 + 2.0 * position,
            "peat_overlap_pct": 0.4 * position,
        }
    )
    feature_columns = ["wetland_suitability", "opportunity_cost_proxy"]
    reference_summary = None
    print("Using deterministic synthetic context; real data and reference are not read.")
else:
    required_inputs = {
        "features": FEATURES_PATH,
        "grid": GRID_PATH,
        "Notebook 10 reference summary": REFERENCE_SUMMARY_PATH,
    }
    missing_inputs = [f"{name}: {path}" for name, path in required_inputs.items() if not path.exists()]
    if missing_inputs:
        raise FileNotFoundError(
            "Full historical inputs are required for screen/full profiles:\n"
            + "\n".join(missing_inputs)
        )
    context = pd.read_parquet(FEATURES_PATH)
    reference_summary = pd.read_parquet(REFERENCE_SUMMARY_PATH)
    feature_columns = [name for name in NOTEBOOK_10_FEATURE_COLUMNS if name in context.columns]
    if not feature_columns:
        raise ValueError("No Notebook 10 feature columns were found in the feature table")

print(f"Context rows: {len(context):,}")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

## Six-scenario seed-42 manifest

In [ ]:
manifest = build_baseline_manifest(profile=PROFILE, seeds=[42])
planned_runs = manifest_run_count(manifest)
print(f"Planned optimizer runs: {planned_runs}")
display(manifest.head(6))

## Parallel historical execution

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def report_progress(done, total, status):
    print(f"Progress: {done}/{total} [{status}]", flush=True)

run_results = run_manifest(
    context,
    feature_columns,
    manifest,
    OUTPUT_ROOT,
    PROFILE,
    n_workers=min(N_WORKERS, planned_runs),
    progress=report_progress,
)
failed_runs = run_results.loc[run_results["status"] == "failed"]
if not failed_runs.empty:
    raise RuntimeError(f"Historical execution failed for {len(failed_runs)} run(s)")
display(run_results[["scenario", "seed", "status", "metrics_path"]])

## Reproduction table

In [ ]:
candidate_summary = pd.concat(
    [pd.read_parquet(path) for path in run_results["metrics_path"]],
    ignore_index=True,
)
reproduction_dir = OUTPUT_ROOT / "reproduction"
reproduction_dir.mkdir(parents=True, exist_ok=True)
candidate_summary_path = reproduction_dir / "seed_42_candidate_summary.parquet"
candidate_summary.to_parquet(candidate_summary_path, index=False)

if PROFILE == "full":
    reproduction_table = compare_reference_summary(reference_summary, candidate_summary)
    gate_status = "PASSED" if reproduction_table.empty else "FAILED"
    print(f"FULL REPRODUCTION GATE: {gate_status}")
else:
    reproduction_table = pd.DataFrame(
        [
            {
                "status": "PENDING",
                "reason": "A full-profile seed-42 run is required for scientific comparison.",
            }
        ]
    )
    print("FULL REPRODUCTION GATE: PENDING (test/screen profiles are smoke runs only)")

comparison_path = reproduction_dir / "seed_42_comparison.parquet"
reproduction_table.to_parquet(comparison_path, index=False)
display(reproduction_table)

## Sequential versus parallel benchmark

In [ ]:
benchmark_table = benchmark_manifest(
    context,
    feature_columns,
    manifest.head(2),
    PROFILE,
)
benchmark_dir = OUTPUT_ROOT / "benchmarks"
benchmark_dir.mkdir(parents=True, exist_ok=True)
benchmark_path = benchmark_dir / "seed_42_parallelism.parquet"
benchmark_table.to_parquet(benchmark_path, index=False)
display(benchmark_table)

## Artifact locations

In [ ]:
artifact_locations = pd.DataFrame(
    {
        "artifact": ["run root", "candidate summary", "comparison table", "benchmark"],
        "path": [OUTPUT_ROOT, candidate_summary_path, comparison_path, benchmark_path],
    }
)
display(artifact_locations)

if PROFILE == "full" and not reproduction_table.empty:
    raise AssertionError(
        f"Seed-42 reproduction failed with {len(reproduction_table)} mismatch(es); "
        f"see {comparison_path}"
    )